# InsectSet459: Stream-and-Delete Feature Extraction Pipeline (Google Colab)

This notebook demonstrates the **stream-and-delete** feature extraction architecture:
1. Streams audio recordings directly from Zenodo zip archives via HTTP range requests into memory (`io.BytesIO`).
2. Decodes, chunks, and extracts 146-dimensional handcrafted features.
3. Checkpoints progress per-recording to Drive so runtime disconnects resume without re-downloading.
4. Immediately discards audio buffers, writing only `.npy` feature arrays directly to Google Drive.

**Storage Benefit:**
- 40-Species Stage: ~4.5 MB feature cache on Drive (vs 1.03 GB raw audio).
- Full 141-Species Benchmark: ~46 MB feature cache on Drive (vs 45.77 GB raw audio).
- Zero audio files stored on Google Drive or Colab local disk.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/insect_dl/data/processed/species40')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Target directory on Google Drive: {DRIVE_OUTPUT_DIR}")

## 2. Install Dependencies

In [ ]:
!pip install -q soundfile librosa soxr remotezip pandas requests tqdm scikit-learn

## 3. Clone Repository

In [ ]:
%cd /content
if not os.path.exists('insect-audio-dl'):
    !git clone https://github.com/ITWhizHarsh/insect-audio-dl.git
%cd /content/insect-audio-dl

## 4. Run Streaming Feature Extraction with Checkpoint Resume

In [ ]:
# If disconnected mid-run, re-executing this cell automatically resumes from the last completed file
!python scripts/colab_stream_features.py \
    --manifest data/meta/species40_manifest.csv \
    --drive-dir /content/drive/MyDrive/insect_dl/data/processed/species40 \
    --workers 4

## 5. Parity Verification (vs Reference 40-Species Benchmark)

Compares the streamed feature arrays against reference arrays, reporting library versions and maximum absolute differences.

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import librosa
import soundfile
import soxr
import sklearn

print("=== Environment & Library Versions ===")
print(f"python:       {sys.version.split()[0]}")
print(f"torch:        {torch.__version__}")
print(f"numpy:        {np.__version__}")
print(f"pandas:       {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"librosa:      {librosa.__version__}")
print(f"soundfile:    {soundfile.__version__}")
print(f"soxr:         {soxr.__version__}")

splits = ['Train', 'Validation', 'Test']
print("\n=== Parity Check vs Reference Arrays ===")
print(f"{'Split':<12} | {'Shape':<14} | {'np.array_equal':<15} | {'Max Abs Diff':<12}")
print("-" * 60)

for split in splits:
    colab_hand = np.load(DRIVE_OUTPUT_DIR / f"{split}_hand.npy")
    ref_path = Path(f"data/processed/species40/{split}_hand.npy")
    if ref_path.exists():
        ref_hand = np.load(ref_path)
        match = np.array_equal(colab_hand, ref_hand)
        max_abs = float(np.max(np.abs(colab_hand - ref_hand)))
        print(f"{split:<12} | {str(colab_hand.shape):<14} | {str(match):<15} | {max_abs:.2e}")
    else:
        print(f"{split:<12} | {str(colab_hand.shape):<14} | [Ref not in local path]")

## 6. Train MLP Baseline on Google Drive Data

In [ ]:
!python -m src.train \
    --model mlp \
    --data /content/drive/MyDrive/insect_dl/data/processed/species40 \
    --epochs 150 \
    --batch 64 \
    --wd 1e-4 \
    --seed 42 \
    --eval-test \
    --output /content/drive/MyDrive/insect_dl/results/mlp_species40_s42.json